In [ ]:
#return稍低，sharpe ratio較高，最大回撤較小，穩健。
import pandas as pd
import vectorbt as vbt
import numpy as np
from indicators import Indicators
import yfinance_fetcher
import Quadrant
import plotly as plotly

# 1. 讀取標的
ticker_list = [line.strip() for line in open('Mystocks.txt', encoding='utf-16') if line.strip()]

def quadrant_analysis(ticker):
    ind = Indicators()
    fetcher = yfinance_fetcher.Yfinance_fetcher()
    ana = Quadrant.MarketQuadrantAnalyzer()
    
    # 抓取資料 (確保長度足以支撐 225 天暖機 + 2 年回測)
    df = fetcher.fetch(ticker, period="3y") 
    df_ind = ind.get_indicators(df)
    df_final = ana.analyze_dataframe(df_ind)
    df_final = ana.attach_descriptions(df_final)
    return df_final

# 準備存儲所有標的績效的清單
all_stats = []
portfolio_values = []  # + 新增一個清單，用來收集每檔股票的每日資金變化

for ticker in ticker_list:
    
    # 執行妳原有的分析模組
    df = quadrant_analysis(ticker)
    
    # --- 2. 定義進出場訊號 (向量化運算) ---
    
    # 進場
    q4_prev = ((df['x_score'].shift(1) > 0) & (df['y_score'].shift(1) < 0)&
               (df['x_score'].shift(2) > 0) & (df['y_score'].shift(2) < 0)&
               (df['x_score'].shift(3) > 0) & (df['y_score'].shift(3) < 0)&
               (df['x_score'].shift(4) > 0) & (df['y_score'].shift(4) < 0)&
               (df['x_score'].shift(5) > 0) & (df['y_score'].shift(5) < 0))
    q1_prev = ((df['x_score'].shift(1) > 0) & (df['y_score'].shift(1) < 0)&
               (df['x_score'].shift(2) > 0) & (df['y_score'].shift(2) < 0))
    q4_curr = (df['x_score'] > 0) & (df['y_score'] <= 0)
    q1_curr = (df['x_score'] > 0) & (df['y_score'] >= 0)
    entries = q4_prev & q4_curr | q1_prev & q1_curr
    
    # 出場：Quadrant1
    # 止損：開始收縮
    trend_exit_q1 = ((df['x_score']<df['x_score'].shift(1))&
                     (df['x_score'].shift(1)<df['x_score'].shift(2))&
                     df['x_score'] < 0) 
    # 止盈：動能收縮
    profit_exit_q1 = ((df['y_score']<df['y_score'].shift(5))&
                      (df['y_score']<df['y_score'].shift(10)))
    # 出場：Quadrant4
    # 止損：開始收縮
    trend_exit_q4 = ((df['x_score']<df['x_score'].shift(5))&
                     (df['x_score']<df['x_score'].shift(10))&
                     df['x_score'] < 0)
    # 止盈
    profit_exit_q4 = ((df['y_score'] > 0) & (df['y_score'] - df['y_score'].shift(1) >= 1))
    # 組合出場訊號：delta x > 1 連續兩天 OR x 轉負 (轉向收縮)
    dx = df['x_score'].diff()
    stagnate_exit = (dx < 0) & (dx.shift(1) < 0)

    exits = trend_exit_q1 | trend_exit_q4 | profit_exit_q4 | stagnate_exit
    
    # --- 3. 處理暖機期與訊號清理 ---
    
    # 剔除前 225 天，避免指標未計算完成的偏誤
    entries[:225] = False
    exits[:225] = False

    # --- 4. 使用 vectorbt 執行回測 ---
    
    pf = vbt.Portfolio.from_signals(
        df['close'], 
        entries, 
        exits,
        fees=0.001,
        freq='1D', 
        init_cash=100000,
        tp_stop=0.1,  # 停利
        #sl_stop=0.1   # 停損
    )
    
    # 收集統計數據
    stats = pf.stats()
    stats['Ticker'] = ticker
    all_stats.append(stats)
    
    # 如果想看個別標的的圖表，可以取消註解下一行
    #pf.plot().show()

    # 收集這檔股票的每日資金水位
    portfolio_values.append(pf.value())


# 5. ---彙整結果並輸出---
# 1. 輸出個別標的總表
final_perf_df = pd.concat(all_stats, axis=1).T
final_perf_df.to_excel("backtest_summary.xlsx")
print("\n--- 個別標的回測彙整 ---")
display(final_perf_df[['Ticker', 'Start Value', 'End Value', 'Total Return [%]', 'Max Drawdown [%]', 'Sharpe Ratio']])

# 2. 計算並輸出整體投資組合績效
# 將所有股票的每日資金加總，形成一條總權益曲線
total_equity = pd.concat(portfolio_values, axis=1).sum(axis=1)

# 計算每日的百分比報酬率，並剔除第一天的空值
overall_returns = total_equity.pct_change().dropna()

# 使用 vectorbt 的 returns 模組計算統計數據 (必須指定 freq='1D')
overall_stats = overall_returns.vbt.returns(freq='1D').stats()

print("\n--- 整體投資組合總績效 (All-in-One Portfolio) ---")
print(overall_stats)

# 繪製並顯示資金曲線
# 使用 vbt 的繪圖功能，這會產生一個互動式的 Plotly 圖表
fig = total_equity.vbt.plot(
    trace_kwargs=dict(name='Total Equity', line=dict(color='blue')))

overall_returns.vbt.returns(freq='1D').plot().show()

# 設定圖表標題
fig.update_layout(title_text='整體投資組合資金曲線', xaxis_title='日期', yaxis_title='總價值')

# 顯示圖表
fig.show()

In [ ]:
#reurn較高，sharpe ratio稍低，最大回撤較大。
import pandas as pd
import vectorbt as vbt
import numpy as np
from indicators import Indicators
import yfinance_fetcher
import Quadrant

# 1. 讀取標的
ticker_list = [line.strip() for line in open('Mystocks.txt', encoding='utf-16') if line.strip()]

def quadrant_analysis(ticker):
    ind = Indicators()
    fetcher = yfinance_fetcher.Yfinance_fetcher()
    ana = Quadrant.MarketQuadrantAnalyzer()
    
    # 抓取資料 (確保長度足以支撐 225 天暖機 + 2 年回測)
    df = fetcher.fetch(ticker, period="3y") 
    df_ind = ind.get_indicators(df)
    df_final = ana.analyze_dataframe(df_ind)
    df_final = ana.attach_descriptions(df_final)
    return df_final

# 準備存儲所有標的績效的清單
all_stats = []
portfolio_values = []  # + 新增一個清單，用來收集每檔股票的每日資金變化

for ticker in ticker_list:
    
    # 執行妳原有的分析模組
    df = quadrant_analysis(ticker)
    
    # --- 2. 定義進出場訊號 (向量化運算) ---
    
    # 進場
    q4_prev = ((df['x_score'].shift(1) > 0) & (df['y_score'].shift(1) < 0)&
               (df['x_score'].shift(2) > 0) & (df['y_score'].shift(2) < 0)&
               (df['x_score'].shift(3) > 0) & (df['y_score'].shift(3) < 0)&
               (df['x_score'].shift(4) > 0) & (df['y_score'].shift(4) < 0)&
               (df['x_score'].shift(5) > 0) & (df['y_score'].shift(5) < 0))
    q1_prev = ((df['x_score'].shift(1) > 0) & (df['y_score'].shift(1) < 0)&
               (df['x_score'].shift(2) > 0) & (df['y_score'].shift(2) < 0)&
               (df['x_score'].shift(3) <= 0) & (df['y_score'].shift(3) <= 0)&
               (df['x_score'].shift(4) <= 0) & (df['y_score'].shift(4) <= 0))
    q4_curr = (df['x_score'] > 0) & (df['y_score'] <= 0)
    q1_curr = (df['x_score'] > 0) & (df['y_score'] >= 0)
    entries = q4_prev & q4_curr | q1_prev & q1_curr
    
    # 出場：Quadrant1
    # 止損：開始收縮
    trend_exit_q1 = ((df['x_score']<df['x_score'].shift(1))&
                     (df['x_score'].shift(1)<df['x_score'].shift(2))&
                     df['x_score'] < 0) 
    # 止盈：動能收縮
    profit_exit_q1 = ((df['y_score']<df['y_score'].shift(5))&
                      (df['y_score']<df['y_score'].shift(10)))
    # 出場：Quadrant4
    # 止損：開始收縮
    trend_exit_q4 = ((df['x_score']<df['x_score'].shift(5))&
                     (df['x_score']<df['x_score'].shift(10))&
                     df['x_score'] < 0)
    # 止盈
    profit_exit_q4 = ((df['y_score'] > 0) & (df['y_score'] - df['y_score'].shift(1) >= 1))
    # 組合出場訊號：delta x > 1 連續兩天 OR x 轉負 (轉向收縮)
    dx = df['x_score'].diff()
    stagnate_exit = (dx < 0) & (dx.shift(1) < 0)

    exits = trend_exit_q1 | trend_exit_q4 | profit_exit_q4 | stagnate_exit
    
    # --- 3. 處理暖機期與訊號清理 ---
    
    # 剔除前 225 天，避免指標未計算完成的偏誤
    entries[:225] = False
    exits[:225] = False
    
    # --- 4. 使用 vectorbt 執行回測 ---
    
    pf = vbt.Portfolio.from_signals(
        df['close'], 
        entries, 
        exits,
        fees=0.001,
        freq='1D', 
        init_cash=100000,
        tp_stop=0.1,  # 停利
        #sl_stop=0.1   # 停損
    )
    
    # 收集統計數據
    stats = pf.stats()
    stats['Ticker'] = ticker
    all_stats.append(stats)
    
    # 如果想看個別標的的圖表，可以取消註解下一行
    #pf.plot().show()

    # 收集這檔股票的每日資金水位
    portfolio_values.append(pf.value())


# 5. ---彙整結果並輸出---
# 1. 輸出個別標的總表
final_perf_df = pd.concat(all_stats, axis=1).T
final_perf_df.to_excel("backtest_summary.xlsx")
print("\n--- 個別標的回測彙整 ---")
display(final_perf_df[['Ticker', 'Start Value', 'End Value', 'Total Return [%]', 'Max Drawdown [%]', 'Sharpe Ratio']])

# 2. 計算並輸出整體投資組合績效
# 將所有股票的每日資金加總，形成一條總權益曲線
total_equity = pd.concat(portfolio_values, axis=1).sum(axis=1)

# 計算每日的百分比報酬率，並剔除第一天的空值
overall_returns = total_equity.pct_change().dropna()

# 使用 vectorbt 的 returns 模組計算統計數據 (必須指定 freq='1D')
overall_stats = overall_returns.vbt.returns(freq='1D').stats()

print("\n--- 整體投資組合總績效 (All-in-One Portfolio) ---")
print(overall_stats)

In [ ]:
%pip install plotly

In [ ]:
%pip install anywidget

In [ ]:
%pip install --upgrade nbformat

In [ ]:
#2026.3.28 V1
import pandas as pd
import vectorbt as vbt
import numpy as np
from indicators import Indicators
import yfinance_fetcher
import Quadrant

# 1. 讀取標的
ticker_list = [line.strip() for line in open('Mystocks.txt', encoding='utf-16') if line.strip()]

def quadrant_analysis(ticker):
    ind = Indicators()
    fetcher = yfinance_fetcher.Yfinance_fetcher()
    ana = Quadrant.MarketQuadrantAnalyzer()
    
    # 抓取資料 (確保長度足以支撐 225 天暖機 + 2 年回測)
    df = fetcher.fetch(ticker, period="3y") 
    df_ind = ind.get_indicators(df)
    df_final = ana.analyze_dataframe(df_ind)
    df_final = ana.attach_descriptions(df_final)
    return df_final

# 準備存儲所有標的績效的清單
all_stats = []
portfolio_values = []  # + 新增一個清單，用來收集每檔股票的每日資金變化

for ticker in ticker_list:
    # 執行妳原有的分析模組
    df = quadrant_analysis(ticker)
    
    # --- 2. 定義進出場訊號 (向量化運算) ---
    
    # 進場
    q4_prev = ((df['x_score'].shift(1) > 0) & (df['y_score'].shift(1) < 0)&
               (df['x_score'].shift(2) > 0) & (df['y_score'].shift(2) < 0)&
               (df['x_score'].shift(3) > 0) & (df['y_score'].shift(3) < 0))
    q1_prev = ((df['x_score'].shift(1) > 0) & (df['y_score'].shift(1) < 0)&
               (df['x_score'].shift(2) > 0) & (df['y_score'].shift(2) < 0)&
               (df['x_score'].shift(3) <= 0) & (df['y_score'].shift(3) <= 0))
    q4_curr = (df['x_score'] > 0) & (df['y_score'] <= 0)
    q1_curr = (df['x_score'] > 0) & (df['y_score'] >= 0)
    entries = q4_prev & q4_curr | q1_prev & q1_curr
    
    # 出場：Quadrant1
    # 止損：開始收縮
    trend_exit_q1 = ((df['x_score']<df['x_score'].shift(1))&
                     (df['x_score'].shift(1)<df['x_score'].shift(2))&
                     df['x_score'] < 0) 
    # 止盈：動能收縮
    profit_exit_q1 = ((df['y_score']<df['y_score'].shift(5))&
                      (df['y_score']<df['y_score'].shift(10)))
    # 出場：Quadrant4
    # 止損：開始收縮
    trend_exit_q4 = ((df['x_score']<df['x_score'].shift(5))&
                     (df['x_score']<df['x_score'].shift(10))&
                     df['x_score'] < 0)
    # 止盈
    profit_exit_q4 = ((df['y_score'] > 0) & (df['y_score'] - df['y_score'].shift(1) >= 1))
    # 組合出場訊號：delta x > 1 連續兩天 OR x 轉負 (轉向收縮)
    dx = df['x_score'].diff()
    stagnate_exit = (dx < 0) & (dx.shift(1) < 0)
    # 象限出場
    quadrant_exit_q2 = (df['x_score'] < 0) & (df['y_score'] > 0) & (df['x_score'].shift(1) < 0) & (df['y_score'].shift(1) > 0)
    quadrant_exit_q3 = (df['x_score'] < 0) & (df['y_score'] < 0) & (df['x_score'].shift(1) < 0) & (df['y_score'].shift(1) < 0)

    exits = trend_exit_q1 | trend_exit_q4 | profit_exit_q4 | stagnate_exit | quadrant_exit_q3 | quadrant_exit_q2
    
    # --- 3. 處理暖機期與訊號清理 ---
    
    # 剔除前 225 天，避免指標未計算完成的偏誤
    entries[:225] = False
    exits[:225] = False

    # --- 4. 使用 vectorbt 執行回測 ---
    
    pf = vbt.Portfolio.from_signals(
        df['close'], 
        entries, 
        exits,
        fees=0.001425,   # 手續費 (0.1425%)
        freq='1D', 
        init_cash=100000,       
        slippage=0.001,
        tp_stop=0.1  # 停利
        #sl_stop=0.1   # 停損
    )
    
    # 收集統計數據
    stats = pf.stats()
    stats['Ticker'] = ticker
    all_stats.append(stats)
    
    # 如果想看個別標的的圖表，可以取消註解下一行
    #pf.plot().show()

    # 收集這檔股票的每日資金水位
    portfolio_values.append(pf.value())


# 5. ---彙整結果並輸出---
# 1. 輸出個別標的總表
final_perf_df = pd.concat(all_stats, axis=1).T
final_perf_df.to_excel("backtest_summary.xlsx")
print("\n--- 個別標的回測彙整 ---")
display(final_perf_df[['Ticker', 'Start Value', 'End Value', 'Total Return [%]', 'Max Drawdown [%]', 'Sharpe Ratio']])

# 2. 計算並輸出整體投資組合績效
# 將所有股票的每日資金加總，形成一條總權益曲線
total_equity = pd.concat(portfolio_values, axis=1).sum(axis=1)

# 計算每日的百分比報酬率，並剔除第一天的空值
overall_returns = total_equity.pct_change().dropna()

# 使用 vectorbt 的 returns 模組計算統計數據 (必須指定 freq='1D')
overall_stats = overall_returns.vbt.returns(freq='1D').stats()

print("\n--- 整體投資組合總績效 (All-in-One Portfolio) ---")
print(overall_stats)

# 繪製並顯示資金曲線
# 使用 vbt 的繪圖功能，這會產生一個互動式的 Plotly 圖表
fig = total_equity.vbt.plot(
    trace_kwargs=dict(name='Total Equity', line=dict(color='blue')))

overall_returns.vbt.returns(freq='1D').plot().show()

# 設定圖表標題
fig.update_layout(title_text='整體投資組合資金曲線', xaxis_title='日期', yaxis_title='總價值')

# 顯示圖表
fig.show()

In [ ]:
#2026.3.28 V2
import pandas as pd
import vectorbt as vbt
import numpy as np
from indicators import Indicators
import yfinance_fetcher
import Quadrant

# 1. 讀取標的
ticker_list = [line.strip() for line in open('Mystocks.txt', encoding='utf-16') if line.strip()]

def quadrant_analysis(ticker):
    ind = Indicators()
    fetcher = yfinance_fetcher.Yfinance_fetcher()
    ana = Quadrant.MarketQuadrantAnalyzer()
    
    # 抓取資料 (確保長度足以支撐 225 天暖機 + 2 年回測)
    df = fetcher.fetch(ticker, period="3y") 
    df_ind = ind.get_indicators(df)
    df_final = ana.analyze_dataframe(df_ind)
    df_final = ana.attach_descriptions(df_final)
    return df_final

# 準備存儲所有標的績效的清單
all_stats = []
portfolio_values = []  # + 新增一個清單，用來收集每檔股票的每日資金變化

for ticker in ticker_list:
    # 執行妳原有的分析模組
    df = quadrant_analysis(ticker)
    
    # --- 2. 定義進出場訊號 (向量化運算) ---
    
    # 進場
    q4_prev = ((df['x_score'].shift(1) > 0) & (df['y_score'].shift(1) < 0)&
               (df['x_score'].shift(2) > 0) & (df['y_score'].shift(2) < 0)&
               (df['x_score'].shift(3) > 0) & (df['y_score'].shift(3) < 0)&
               (df['x_score'].shift(4) > 0) & (df['y_score'].shift(4) < 0)&
               (df['x_score'].shift(5) > 0) & (df['y_score'].shift(5) < 0))
    q1_prev = ((df['x_score'].shift(1) > 0) & (df['y_score'].shift(1) < 0)&
               (df['x_score'].shift(2) > 0) & (df['y_score'].shift(2) < 0))
    q4_curr = (df['x_score'] > 0) & (df['y_score'] <= 0)
    q1_curr = (df['x_score'] > 0) & (df['y_score'] >= 0)
    entries = q4_prev & q4_curr | q1_prev & q1_curr
    
    # 出場：Quadrant1
    # 止損：開始收縮
    trend_exit_q1 = ((df['x_score']<df['x_score'].shift(1))&
                     (df['x_score'].shift(1)<df['x_score'].shift(2))&
                     df['x_score'] < 0) 
    # 止盈：動能收縮
    profit_exit_q1 = ((df['y_score']<df['y_score'].shift(5))&
                      (df['y_score']<df['y_score'].shift(10)))
    # 出場：Quadrant4
    # 止損：開始收縮
    trend_exit_q4 = ((df['x_score']<df['x_score'].shift(5))&
                     (df['x_score']<df['x_score'].shift(10))&
                     df['x_score'] < 0)
    # 止盈
    profit_exit_q4 = ((df['y_score'] > 0) & (df['y_score'] - df['y_score'].shift(1) >= 1))
    # 組合出場訊號：delta x > 1 連續兩天 OR x 轉負 (轉向收縮)
    dx = df['x_score'].diff()
    stagnate_exit = (dx < 0) & (dx.shift(1) < 0)
    # 象限出場
    quadrant_exit_q2 = (df['x_score'] < 0) & (df['y_score'] > 0) & (df['x_score'].shift(1) < 0) & (df['y_score'].shift(1) > 0)
    quadrant_exit_q3 = (df['x_score'] < 0) & (df['y_score'] < 0) & (df['x_score'].shift(1) < 0) & (df['y_score'].shift(1) < 0)

    exits = trend_exit_q1 | trend_exit_q4 | profit_exit_q4 | stagnate_exit | quadrant_exit_q3 | quadrant_exit_q2
    
    # --- 3. 處理暖機期與訊號清理 ---
    
    # 剔除前 225 天，避免指標未計算完成的偏誤
    entries[:225] = False
    exits[:225] = False

    # --- 4. 使用 vectorbt 執行回測 ---
    
    pf = vbt.Portfolio.from_signals(
        df['close'], 
        entries, 
        exits,
        fees=0.001425,   # 手續費 (0.1425%)
        freq='1D', 
        init_cash=100000,       
        slippage=0.001,
        tp_stop=0.1  # 停利
        #sl_stop=0.1   # 停損
    )
    
    # 收集統計數據
    stats = pf.stats()
    stats['Ticker'] = ticker
    all_stats.append(stats)
    
    # 如果想看個別標的的圖表，可以取消註解下一行
    #pf.plot().show()

    # 收集這檔股票的每日資金水位
    portfolio_values.append(pf.value())


# 5. ---彙整結果並輸出---
# 1. 輸出個別標的總表
final_perf_df = pd.concat(all_stats, axis=1).T
final_perf_df.to_excel("backtest_summary.xlsx")
print("\n--- 個別標的回測彙整 ---")
display(final_perf_df[['Ticker', 'Start Value', 'End Value', 'Total Return [%]', 'Max Drawdown [%]', 'Sharpe Ratio']])

# 2. 計算並輸出整體投資組合績效
# 將所有股票的每日資金加總，形成一條總權益曲線
total_equity = pd.concat(portfolio_values, axis=1).sum(axis=1)

# 計算每日的百分比報酬率，並剔除第一天的空值
overall_returns = total_equity.pct_change().dropna()

# 使用 vectorbt 的 returns 模組計算統計數據 (必須指定 freq='1D')
overall_stats = overall_returns.vbt.returns(freq='1D').stats()

print("\n--- 整體投資組合總績效 (All-in-One Portfolio) ---")
print(overall_stats)

# 繪製並顯示資金曲線
# 使用 vbt 的繪圖功能，這會產生一個互動式的 Plotly 圖表
fig = total_equity.vbt.plot(
    trace_kwargs=dict(name='Total Equity', line=dict(color='blue')))

overall_returns.vbt.returns(freq='1D').plot().show()

# 設定圖表標題
fig.update_layout(title_text='整體投資組合資金曲線', xaxis_title='日期', yaxis_title='總價值')

# 顯示圖表
fig.show()